In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, TimestampType, StringType, ArrayType
)
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("traffice-analysis") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 18:53:49 WARN Utils: Your hostname, HP-OMEN-WIN, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/22 18:53:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 18:53:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

# Read raw TSV — event_list and product_list ingested as StringType first
raw_schema = StructType([
    StructField("hit_time_gmt",  LongType(),      False),
    StructField("date_time",      TimestampType(), False),
    StructField("user_agent",     StringType(),    True),
    StructField("ip",             StringType(),    True),
    StructField("event_list",     StringType(),    True),
    StructField("geo_city",       StringType(),    True),
    StructField("geo_region",     StringType(),    True),
    StructField("geo_country",    StringType(),    True),
    StructField("pagename",       StringType(),    True),
    StructField("page_url",       StringType(),    True),
    StructField("product_list",   StringType(),    True),
    StructField("referrer",       StringType(),    True),
])

df = (
    spark.read.csv(
        "./data/data.sql",
        schema=raw_schema,
        sep="\t",
        header=True,
        timestampFormat="yyyy-MM-dd HH:mm:ss",
    )
    # event_list: comma-separated event codes → Array[String]
    .withColumn(
        "event_list",
        F.when(F.col("event_list").isNotNull(),
               F.split(F.col("event_list"), ","))
         .otherwise(F.lit(None))
    )
    # product_list: comma-separated product entries → Array[String]
    .withColumn(
        "product_list",
        F.when(F.col("product_list").isNotNull(),
               F.split(F.col("product_list"), ","))
         .otherwise(F.lit(None))
    )
)

df.printSchema()

root
 |-- hit_time_gmt: long (nullable = true)
 |-- date_time: timestamp (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- ip: string (nullable = true)
 |-- event_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- geo_city: string (nullable = true)
 |-- geo_region: string (nullable = true)
 |-- geo_country: string (nullable = true)
 |-- pagename: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- referrer: string (nullable = true)



In [3]:
df.show()

+------------+-------------------+--------------------+-------------+----------+--------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+
|hit_time_gmt|          date_time|          user_agent|           ip|event_list|      geo_city|geo_region|geo_country|            pagename|            page_url|        product_list|            referrer|
+------------+-------------------+--------------------+-------------+----------+--------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+
|  1254033280|2009-09-27 06:34:40|Mozilla/5.0 (Wind...|  67.98.123.1|      NULL|         Salem|        OR|         US|                Home|http://www.esshop...|                NULL|http://www.google...|
|  1254033379|2009-09-27 06:36:19|Mozilla/5.0 (Maci...|   23.8.61.21|       [2]|     Rochester|        NY|         US|        Zune - 32 GB|http://www.esshop...|[Electronics;Zune...|http://